# Grade Distribution Per AE — Stacked Bars

For each adverse event, a stacked bar of CTCAE grades 1–5 (bottom-to-top), colored
by the unified `GRADE_COLORS` severity palette. Six panels share the Fig 1B toxicity
order, each with its own y-axis so smaller AEs remain readable.

**Cohort/grade logic:** reads the current per-toxicity master tables (confirmed single-toxicity, no
arbitrary overwrite — avoiding the "arbitrary AE" mislabeling issue where a patient's single AE slot
on a line could be silently overwritten if they had multiple AEs in the same 90-day window). The
cohort is restricted to **first line of therapy only**, using the same `line1` construction as S2C
(first LOT row per patient, requiring a valid `lot_start` and a finite positive `t_cutoff_lot`), so
the denominator and the line-1 definition match across supplementary panels. Each patient
contributes their `ae_grade` on that line-1 row.

**Formatting (Nature compliance):** Arial only (hard-fails if not resolved), `pdf.fonttype=42`,
matches the sizing/style already established for other scrupts 

In [ ]:
%matplotlib inline

import re, os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 450,
})
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import MaxNLocator

# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")

def standardize_mrn(mrn):
    if pd.isna(mrn):
        return None
    try:
        digits = re.findall(r"\d+", str(mrn).strip().strip("'\""))
        return str(int(digits[0])).zfill(8) if digits else None
    except (ValueError, TypeError):
        return None


## Paths and constants

Per-toxicity master tables (same source as the time-dependent validation panel). `grade0` tier =
this project's "Grade 1+" convention (confirmed empirically: grades 1-4 present in that tier).


In [ ]:
NOTEBOOK_DIR = os.getcwd()
FIGURES_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
DATA_DIR = os.path.join(FIGURES_DIR, 'figures_data', 'figure 2', 'data')
TOX_TABLE_DIR = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026')  # update if files move into DATA_DIR directly
RESULTS_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results', 'supp', 'S2B_Grade_Histogram'))
os.makedirs(RESULTS_DIR, exist_ok=True)

LLM_PATIENT_PATH = os.path.join(DATA_DIR, 'llm_calls_patient_level_84k.csv')
PDF_OUT = os.path.join(RESULTS_DIR, 'S2B_Grade_Histogram.pdf')
CSV_OUT = os.path.join(RESULTS_DIR, 'S2B_Grade_Histogram_Values.csv')

TOXICITY_COLUMNS = ['liver_toxicity', 'hypothyroidism', 'pneumonitis',
                    'colitis', 'adrenal_insufficiency', 'hyperthyroidism']
TOXICITY_DISPLAY = {
    'pneumonitis': 'Pneumonitis', 'adrenal_insufficiency': 'Adrenal Insufficiency',
    'liver_toxicity': 'Liver Toxicity', 'colitis': 'Colitis',
    'hyperthyroidism': 'Hyperthyroidism', 'hypothyroidism': 'Hypothyroidism',
}
GRADE_ORDER = [1, 2, 3, 4, 5]
GRADE_COLORS = {1: '#FEE391', 2: '#F4A02C', 3: '#F4663A', 4: '#B30000', 5: '#54278F'}
GRADE_LABELS = {1: 'Grade 1', 2: 'Grade 2', 3: 'Grade 3', 4: 'Grade 4', 5: 'Grade 5'}
GRADE_TIER = 'grade0'  # == Grade 1+, confirmed empirically

TOX_TABLE_PATHS = {
    tox: os.path.join(TOX_TABLE_DIR, f'llm84k_{tox}_{GRADE_TIER}_20260630.csv')
    for tox in TOXICITY_COLUMNS
}
for p in list(TOX_TABLE_PATHS.values()) + [LLM_PATIENT_PATH]:
    print(('FOUND   ' if os.path.exists(p) else 'MISSING '), p)

# Canonical line-1 source, identical to S2C (Sex_Prevalance_S2C.ipynb).
# Every per-toxicity table shares the same LOT structure; pneumonitis is the reference
# so the line-1 definition is fixed once and applied identically to all six toxicities.
COVARS_PATH = TOX_TABLE_PATHS['pneumonitis']

## Cohort restriction and line-1 grade, per toxicity

Restrict to patients with LLM predictions (cohort membership, matching every other panel), then
restrict to **first line of therapy only**, using the same `line1` construction as S2C: the first
LOT row per patient (`idxmin` on `lot`, not `.groupby().first()`), requiring a non-null `lot_start`
and a finite, positive `t_cutoff_lot`. The resulting `(mrn, line1_lot)` pair is then inner-joined
onto each per-toxicity table, so the line-1 definition is fixed once from a single canonical source
rather than re-derived independently inside each toxicity file (which would let a patient whose
LOT-1 row is absent from one table contribute their LOT-2 grade to that toxicity).

The grade counted is the patient's `ae_grade` on that line-1 row. `max` is retained only to collapse
duplicate `(mrn, lot)` rows should any exist; with one row per patient it is a no-op.

In [ ]:
llm_patients = pd.read_csv(LLM_PATIENT_PATH, encoding='latin-1', low_memory=False)
llm_patients['mrn'] = llm_patients['mrn'].apply(standardize_mrn)
llm_patients = llm_patients[llm_patients['mrn'].notna()].copy()
cohort_mrns = set(llm_patients['mrn'].drop_duplicates())
print(f'{len(cohort_mrns):,} unique patients with LLM predictions')

# ---- line1: first LOT per patient (identical construction to S2C) ----
covars = pd.read_csv(COVARS_PATH, low_memory=False,
                     usecols=['mrn', 'lot', 'lot_start', 't_cutoff_lot'])
covars['mrn'] = covars['mrn'].apply(standardize_mrn)
covars = covars[covars['mrn'].notna()].copy()
covars['lot'] = pd.to_numeric(covars['lot'], errors='coerce')
covars['lot_start'] = pd.to_datetime(covars['lot_start'], errors='coerce')
covars['censor_days'] = pd.to_numeric(covars['t_cutoff_lot'], errors='coerce')

covars_valid = covars[covars['lot_start'].notna() & covars['lot'].notna()].copy()
idx = covars_valid.sort_values(['mrn', 'lot']).groupby('mrn')['lot'].idxmin()
line1 = (covars_valid.loc[idx, ['mrn', 'lot', 'lot_start', 'censor_days']]
         .rename(columns={'lot': 'line1_lot', 'lot_start': 'line1_start'}))
line1 = line1[np.isfinite(line1['censor_days']) & (line1['censor_days'] > 0)].copy()
line1 = line1[line1['mrn'].isin(cohort_mrns)].copy()
print(f'{len(line1):,} patients with line 1 dates and valid censoring (cohort-restricted)')
print(f"  line-1 LOT value distribution: {line1['line1_lot'].value_counts().sort_index().to_dict()}")


def line1_grade_per_patient(tox):
    """Patient's ae_grade on their line-1 row for this toxicity. Returns a Series indexed by mrn."""
    df = pd.read_csv(TOX_TABLE_PATHS[tox], low_memory=False,
                     usecols=['mrn', 'lot', 'ae_toxicity', 'ae_grade'])
    df['mrn'] = df['mrn'].apply(standardize_mrn)
    df = df[df['mrn'].notna()].copy()
    df['lot'] = pd.to_numeric(df['lot'], errors='coerce')
    df = df[df['lot'].notna()].copy()

    # First-line filter: inner-join on the canonical (mrn, line-1 LOT) pair.
    df = df.merge(line1[['mrn', 'line1_lot']], on='mrn', how='inner')
    df = df[df['lot'] == df['line1_lot']].copy()

    df = df[df['ae_toxicity'].notna()].copy()  # confirmed single-toxicity per file, no mapping needed
    df['ae_grade'] = pd.to_numeric(df['ae_grade'], errors='coerce')
    df = df[df['ae_grade'].notna() & df['ae_grade'].isin(GRADE_ORDER)].copy()
    if df.empty:
        return pd.Series(dtype=int)
    return df.groupby('mrn')['ae_grade'].max().astype(int)  # collapses duplicate (mrn, lot) rows only


grade_by_tox = {tox: line1_grade_per_patient(tox) for tox in TOXICITY_COLUMNS}
for tox in TOXICITY_COLUMNS:
    s = grade_by_tox[tox]
    print(f'  {TOXICITY_DISPLAY[tox]}: {len(s):,} patients with a line-1 grade '
          f'({s.value_counts().reindex(GRADE_ORDER, fill_value=0).to_dict()})')

# Sanity check: every patient counted must be in the line-1 denominator, one grade each.
line1_mrns = set(line1['mrn'])
for tox, s in grade_by_tox.items():
    assert set(s.index).issubset(line1_mrns), f'{tox}: patients outside the line-1 cohort'
    assert s.index.is_unique, f'{tox}: duplicate mrn after line-1 restriction'
print('Line-1 restriction verified for all toxicities.')

## Plot — one stacked bar per toxicity, independent y-axes

In [ ]:
ae_list = TOXICITY_COLUMNS

counts = {
    tox: {g: int((grade_by_tox[tox] == g).sum()) for g in GRADE_ORDER}
    for tox in ae_list
}

def nice_top(n):
    """Smallest 1-2-2.5-3-4-5-6-8-10 * 10^k that is >= n, so 0 / mid / top ticks stay round."""
    import math
    n = max(float(n), 1.0)
    exp = math.floor(math.log10(n))
    base = 10 ** exp
    frac = n / base
    for f in (1, 1.5, 2, 2.5, 3, 4, 5, 6, 8, 10):
        if f + 1e-9 >= frac:
            return f * base
    return 10 * base

def make_count_formatter(show_zero):
    def _fmt(x, _pos):
        if abs(x) < 1e-8:
            return '0' if show_zero else ''
        if abs(x) >= 1000:
            v = x / 1000.0
            return f'{v:.0f}k' if abs(v - round(v)) < 1e-6 else f'{v:.1f}k'
        return f'{int(round(x))}'
    return plt.FuncFormatter(_fmt)

def set_axes_position_inches(fig, ax, left_in, top_in, width_in, height_in):
    fw, fh = fig.get_size_inches()
    ax.set_position([
        left_in / fw,
        1 - (top_in + height_in) / fh,
        width_in / fw,
        height_in / fh,
    ])

# Same canvas and plot-area box as Cancer_Type_Treatment_S2A so the origin
# and x-axis baseline line up in the composite figure.
# S2A measured: left=1.03 right=0.08 top=0.08 bottom=0.93, plot 2.49 x 1.29
FIG_WIDTH_IN     = 3.6
FIG_HEIGHT_IN    = 2.3
LEFT_MARGIN_IN   = 1.03
RIGHT_MARGIN_IN  = 0.08
BOTTOM_MARGIN_IN = 0.93
TOP_MARGIN_IN    = 0.22   # extra strip for the Grade legend; x-axis stays at 0.93 in
PLOT_WIDTH_IN    = FIG_WIDTH_IN  - LEFT_MARGIN_IN - RIGHT_MARGIN_IN
PLOT_HEIGHT_IN   = FIG_HEIGHT_IN - TOP_MARGIN_IN  - BOTTOM_MARGIN_IN

N = 6
GAP_IN = 0.20
ax_w = (PLOT_WIDTH_IN - (N - 1) * GAP_IN) / N

fig = plt.figure(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))
axes = []
for i in range(N):
    ax = fig.add_axes([0, 0, 1, 1])
    axes.append(ax)
    set_axes_position_inches(
        fig, ax,
        left_in=LEFT_MARGIN_IN + i * (ax_w + GAP_IN),
        top_in=TOP_MARGIN_IN,
        width_in=ax_w,
        height_in=PLOT_HEIGHT_IN,
    )

for i, tox in enumerate(ae_list):
    ax = axes[i]
    bottom = 0.0
    total = sum(counts[tox].values())
    for g in GRADE_ORDER:
        n = counts[tox][g]
        ax.bar(
            0, n, bottom=bottom, color=GRADE_COLORS[g], edgecolor='white',
            linewidth=0.5, width=0.75,
            label=GRADE_LABELS[g] if i == 0 else None,
        )
        bottom += n
    ax.set_xlim(-0.55, 0.55)
    ax.set_xticks([0])
    ax.set_xticklabels(
        [TOXICITY_DISPLAY[tox]], rotation=30, ha='right',
        rotation_mode='anchor', fontsize=6,
    )
    top = nice_top(total)
    ax.set_ylim(0, top)
    ax.set_yticks([0, top / 2, top])
    ax.yaxis.set_major_formatter(make_count_formatter(show_zero=(i == 0)))
    ax.tick_params(axis='y', labelsize=6)
    ax.tick_params(axis='x', labelsize=6, length=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(True)
    ax.spines['bottom'].set_visible(True)
    ax.set_axisbelow(False)

# Right-align y-tick labels to each spine so 5k / 1.5k / 100 share a column.
fig.canvas.draw()
for ax in axes:
    for lab in ax.get_yticklabels():
        lab.set_horizontalalignment('right')
        lab.set_verticalalignment('center')

axes[0].set_ylabel('Number of Patients', fontsize=7)

# Continuous x-axis across the full plot box (fills the gaps between panels).
x0 = LEFT_MARGIN_IN / FIG_WIDTH_IN
x1 = (LEFT_MARGIN_IN + PLOT_WIDTH_IN) / FIG_WIDTH_IN
y0 = BOTTOM_MARGIN_IN / FIG_HEIGHT_IN
fig.add_artist(plt.Line2D(
    [x0, x1], [y0, y0], transform=fig.transFigure,
    color='black', linewidth=0.8, clip_on=False, zorder=5, solid_capstyle='butt',
))

fig.text(
    (LEFT_MARGIN_IN + 0.5 * PLOT_WIDTH_IN) / FIG_WIDTH_IN,
    0.06,
    'Adverse event',
    ha='center', va='bottom', fontsize=7, transform=fig.transFigure,
)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, fontsize=5, ncol=5,
    loc='lower center',
    bbox_to_anchor=(
        (LEFT_MARGIN_IN + 0.5 * PLOT_WIDTH_IN) / FIG_WIDTH_IN,
        (FIG_HEIGHT_IN - TOP_MARGIN_IN + 0.02) / FIG_HEIGHT_IN,
    ),
    bbox_transform=fig.transFigure,
    framealpha=0.95, edgecolor='#ccc', fancybox=False,
    handlelength=1.0, handletextpad=0.4, columnspacing=0.8,
)

print(f"margins (in): left={LEFT_MARGIN_IN:.2f} right={RIGHT_MARGIN_IN:.2f} "
      f"top={TOP_MARGIN_IN:.2f} bottom={BOTTOM_MARGIN_IN:.2f}")
print(f"plot area (in): {PLOT_WIDTH_IN:.2f} x {PLOT_HEIGHT_IN:.2f}")
plt.show()


## Save PDF + underlying values CSV


In [ ]:
with PdfPages(PDF_OUT) as pdf:
    pdf.savefig(fig, dpi=450)
plt.close(fig)
print(f'Saved: {os.path.basename(PDF_OUT)}')

records = []
for tox in ae_list:
    for g in GRADE_ORDER:
        records.append({
            'toxicity': tox, 'toxicity_display': TOXICITY_DISPLAY[tox],
            'grade': g, 'grade_label': GRADE_LABELS[g],
            'n_patients': int((grade_by_tox[tox] == g).sum()),
        })
grade_hist_export = pd.DataFrame(records)
grade_hist_export.to_csv(CSV_OUT, index=False)
print(f'Saved: {os.path.basename(CSV_OUT)}')
grade_hist_export
